# Phase 8: Product-Level Trust Aggregation & Evaluation

Aggregate review-level trust scores to product level and evaluate ranking quality.

**Key Metrics:**
- Trust-weighted rating formula
- NDCG@K (Normalized Discounted Cumulative Gain)
- Precision@K
- Comparison vs baselines (raw average, count-weighted)

In [1]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

## 1. Load Data

In [2]:
df = pd.read_csv("../data/processed/reviews_with_predicted_trust.csv")

print(f"Dataset shape: {df.shape}")
print(f"Predicted trust score stats:")
print(df['predicted_trust_score'].describe())

Dataset shape: (719967, 8)
Predicted trust score stats:
count    719967.000000
mean          0.571692
std           0.108702
min           0.146959
25%           0.513470
50%           0.567607
75%           0.612114
max           0.995795
Name: predicted_trust_score, dtype: float64


## 2. Product-Level Trust Aggregation

**Bayesian Average Applied:**
To prevent single-review products from ranking equally with well-reviewed products,
we apply a Bayesian average:

```
score = (n × trust_weighted_rating + m × C) / (n + m)
```

where:
- n = number of reviews for the product
- m = 5 (minimum review threshold)
- C = global mean rating

Products with fewer reviews regress toward the global mean automatically.

In [3]:
df['weighted_rating'] = df['predicted_trust_score'] * df['rating']

product_scores = df.groupby('product_id').agg({
    'weighted_rating': 'sum',
    'predicted_trust_score': 'sum',
    'rating': ['mean', 'count', 'std']
}).reset_index()

product_scores.columns = ['product_id', 'weighted_rating_sum', 'trust_sum', 
                           'avg_rating', 'review_count', 'rating_std']

# Raw trust-weighted rating (before Bayesian adjustment)
product_scores['trust_weighted_rating_raw'] = (
    product_scores['weighted_rating_sum'] / product_scores['trust_sum']
)

# Bayesian average to handle low-review products
# Formula: (n × trust_weighted_rating + m × C) / (n + m)
# where m = minimum review threshold, C = global mean rating
m = 5  # Minimum review threshold
C = product_scores['avg_rating'].mean()  # Global mean rating

product_scores['trust_weighted_rating'] = (
    (product_scores['review_count'] * product_scores['trust_weighted_rating_raw'] + m * C) /
    (product_scores['review_count'] + m)
)

print(f"Product scores shape: {product_scores.shape}")
print(f"Bayesian parameters: m={m}, C={C:.3f}")
print(f"Trust-weighted rating stats (with Bayesian adjustment):")
print(product_scores['trust_weighted_rating'].describe())
print(f"\nComparison for low-review products:")
low_review = product_scores[product_scores['review_count'] <= 3].head(5)
print(f"  Before Bayesian: {low_review['trust_weighted_rating_raw'].mean():.3f}")
print(f"  After Bayesian:  {low_review['trust_weighted_rating'].mean():.3f}")

Product scores shape: (168281, 8)
Bayesian parameters: m=5, C=3.780
Trust-weighted rating stats (with Bayesian adjustment):
count    168281.000000
mean          3.779622
std           0.341623
min           1.100372
25%           3.559732
50%           3.842597
75%           3.983030
max           4.923727
Name: trust_weighted_rating, dtype: float64

Comparison for low-review products:
  Before Bayesian: 4.601
  After Bayesian:  3.884


## 3. Baseline Comparisons

In [4]:
product_scores['baseline_avg_rating'] = product_scores['avg_rating']

min_reviews = product_scores['review_count'].min()
max_reviews = product_scores['review_count'].max()
product_scores['review_weight'] = (
    (product_scores['review_count'] - min_reviews) / (max_reviews - min_reviews)
)
product_scores['baseline_count_weighted'] = (
    product_scores['avg_rating'] * (0.5 + 0.5 * product_scores['review_weight'])
)

print("Baseline scores computed")
print(f"Raw avg rating:        {product_scores['baseline_avg_rating'].mean():.3f} ± {product_scores['baseline_avg_rating'].std():.3f}")
print(f"Count-weighted:        {product_scores['baseline_count_weighted'].mean():.3f} ± {product_scores['baseline_count_weighted'].std():.3f}")
print(f"Trust-weighted:        {product_scores['trust_weighted_rating'].mean():.3f} ± {product_scores['trust_weighted_rating'].std():.3f}")

Baseline scores computed
Raw avg rating:        3.780 ± 1.274
Count-weighted:        1.891 ± 0.638
Trust-weighted:        3.780 ± 0.342


## 4. Ranking Metrics

In [5]:
def dcg_at_k(scores, k=10):
    scores = np.asarray(scores)[:k]
    if len(scores) == 0:
        return 0.0
    return np.sum(scores / np.log2(np.arange(2, len(scores) + 2)))

def ndcg_at_k(y_true, y_pred, k=10):
    sorted_indices = np.argsort(y_pred)[::-1]
    y_true_sorted = y_true[sorted_indices]
    dcg = dcg_at_k(y_true_sorted, k)
    y_true_sorted_ideal = np.sort(y_true)[::-1]
    idcg = dcg_at_k(y_true_sorted_ideal, k)
    if idcg == 0:
        return 0.0
    return dcg / idcg

def precision_at_k(y_true, y_pred, k=10, threshold=4.0):
    sorted_indices = np.argsort(y_pred)[::-1][:k]
    y_true_at_k = y_true[sorted_indices]
    return np.mean(y_true_at_k >= threshold)

print('Ranking metric functions defined.')

Ranking metric functions defined.


## 4. Held-Out Split Ranking Evaluation

**Protocol:** For each product, 80 % of its reviews form the *train split* (used to
compute ranking scores) and the remaining 20 % form the *holdout split*. The ground
truth is the average rating from the **holdout** reviews — an independent signal never
used when building the ranker. This prevents the circular NDCG = 1.0 artifact that
occurs when `y_true` and the ranking signal come from the same source.

Only products with **≥ 5 train reviews** and **≥ 2 holdout reviews** are included,
ensuring stable estimates on both sides of the split.

In [6]:
# ── Held-out split ranking evaluation ────────────────────────────────────
# Shuffle with fixed seed so the 80/20 split is fully reproducible.
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Assign within-product sequential index.
df_shuffled['_rev_idx'] = df_shuffled.groupby('product_id').cumcount()
df_shuffled['_prod_size'] = df_shuffled.groupby('product_id')['product_id'].transform('count')

# 80 / 20 split per product.
split_threshold = 0.8
df_shuffled['_split'] = np.where(
    df_shuffled['_rev_idx'] < (df_shuffled['_prod_size'] * split_threshold).astype(int),
    'train', 'holdout'
)

train_df   = df_shuffled[df_shuffled['_split'] == 'train'].copy()
holdout_df = df_shuffled[df_shuffled['_split'] == 'holdout'].copy()

# ── Build TRAIN-only product aggregations ─────────────────────────────────
train_df['weighted_rating'] = train_df['predicted_trust_score'] * train_df['rating']

train_agg = train_df.groupby('product_id').agg(
    weighted_rating_sum=('weighted_rating', 'sum'),
    trust_sum=('predicted_trust_score', 'sum'),
    train_avg_rating=('rating', 'mean'),
    train_count=('rating', 'count')
).reset_index()

# Raw trust score (before Bayesian adjustment)
train_agg['trust_score_train_raw'] = (
    train_agg['weighted_rating_sum'] / train_agg['trust_sum']
)

# Apply Bayesian average to train scores
m_train = 5
C_train = train_agg['train_avg_rating'].mean()
train_agg['trust_score_train'] = (
    (train_agg['train_count'] * train_agg['trust_score_train_raw'] + m_train * C_train) /
    (train_agg['train_count'] + m_train)
)

# Count-weighted baseline (same formula used in the full aggregation).
min_c = train_agg['train_count'].min()
max_c = train_agg['train_count'].max()
train_agg['count_weight'] = (train_agg['train_count'] - min_c) / max(max_c - min_c, 1)
train_agg['count_weighted_train'] = (
    train_agg['train_avg_rating'] * (0.5 + 0.5 * train_agg['count_weight'])
)

# ── Build HOLDOUT ground-truth ────────────────────────────────────────────
holdout_agg = holdout_df.groupby('product_id').agg(
    holdout_avg_rating=('rating', 'mean'),
    holdout_count=('rating', 'count')
).reset_index()

# ── Merge and filter ──────────────────────────────────────────────────────
eval_df = train_agg.merge(holdout_agg, on='product_id', how='inner')

# Keep products with enough reviews on both sides for stable estimates.
MIN_TRAIN   = 5
MIN_HOLDOUT = 2
eval_df = eval_df[
    (eval_df['train_count'] >= MIN_TRAIN) &
    (eval_df['holdout_count'] >= MIN_HOLDOUT)
].copy()

print(f'Products used for evaluation : {len(eval_df):,}')
print(f'  (filtered from {len(train_agg):,} total products requiring '
      f'>={MIN_TRAIN} train + >={MIN_HOLDOUT} holdout reviews)')
print(f'  Train reviews   : {train_df.shape[0]:,}')
print(f'  Holdout reviews : {holdout_df.shape[0]:,}')

# ── Compute NDCG & Precision against HOLDOUT ground truth ─────────────────
# y_true_holdout is the average rating from reviews the ranker NEVER saw.
y_true_holdout = eval_df['holdout_avg_rating'].values

results_ranking = []
k_values = [5, 10, 20]

for k in k_values:
    ndcg_trust = ndcg_at_k(y_true_holdout, eval_df['trust_score_train'].values, k)
    prec_trust = precision_at_k(y_true_holdout, eval_df['trust_score_train'].values, k)
    ndcg_avg   = ndcg_at_k(y_true_holdout, eval_df['train_avg_rating'].values, k)
    prec_avg   = precision_at_k(y_true_holdout, eval_df['train_avg_rating'].values, k)
    ndcg_count = ndcg_at_k(y_true_holdout, eval_df['count_weighted_train'].values, k)
    prec_count = precision_at_k(y_true_holdout, eval_df['count_weighted_train'].values, k)

    results_ranking.append({
        'K': k,
        'NDCG_Trust': round(ndcg_trust, 6),
        'NDCG_Avg':   round(ndcg_avg,   6),
        'NDCG_Count': round(ndcg_count, 6),
        'Prec_Trust': round(prec_trust, 6),
        'Prec_Avg':   round(prec_avg,   6),
        'Prec_Count': round(prec_count, 6),
    })

import pandas as _pd
ranking_results_df = _pd.DataFrame(results_ranking)

print('\n' + '='*80)
print('HELD-OUT SPLIT RANKING METRICS')
print('Ground truth = holdout avg_rating (independent of ranker inputs)')
print('='*80)
print(ranking_results_df.to_string(index=False))
print('='*80)

print('\nImprovement of Trust-Weighted over Raw Average:')
for _, row in ranking_results_df.iterrows():
    ndcg_imp = (row['NDCG_Trust'] - row['NDCG_Avg']) / max(row['NDCG_Avg'], 1e-9) * 100
    prec_imp = (row['Prec_Trust'] - row['Prec_Avg']) / max(row['Prec_Avg'], 1e-9) * 100
    print(f"  @K={int(row['K']):2d}: NDCG {ndcg_imp:+.2f}%  |  Precision {prec_imp:+.2f}%")

ranking_results_df.to_csv('../results/reports/ranking_metrics.csv', index=False)
print('\nResults saved to ranking_metrics.csv')

Products used for evaluation : 17,013
  (filtered from 75,155 total products requiring >=5 train + >=2 holdout reviews)
  Train reviews   : 467,719
  Holdout reviews : 252,248

HELD-OUT SPLIT RANKING METRICS
Ground truth = holdout avg_rating (independent of ranker inputs)
 K  NDCG_Trust  NDCG_Avg  NDCG_Count  Prec_Trust  Prec_Avg  Prec_Count
 5    0.972668  0.821137    0.915985         1.0      0.60         1.0
10    0.964585  0.859036    0.901130         1.0      0.80         1.0
20    0.956566  0.870327    0.897173         1.0      0.85         1.0

Improvement of Trust-Weighted over Raw Average:
  @K= 5: NDCG +18.45%  |  Precision +66.67%
  @K=10: NDCG +12.29%  |  Precision +25.00%
  @K=20: NDCG +9.91%  |  Precision +17.65%

Results saved to ranking_metrics.csv


## 5. Save Final Product Scores

In [7]:
output_df = product_scores[[
    'product_id', 'review_count', 'avg_rating', 'rating_std',
    'baseline_avg_rating', 'baseline_count_weighted', 'trust_weighted_rating'
]].copy()

output_df.columns = [
    'product_id', 'review_count', 'avg_rating', 'rating_std',
    'score_raw_avg', 'score_count_weighted', 'score_trust_weighted'
]

output_df = output_df.sort_values('score_trust_weighted', ascending=False)
output_df.to_csv('../data/processed/product_trust_scores.csv', index=False)

print(f"Product scores saved: {output_df.shape[0]} products")
print(f"\nTop 20 products by trust-weighted score:")
print(output_df.head(20).to_string(index=False))

Product scores saved: 168281 products

Top 20 products by trust-weighted score:
product_id  review_count  avg_rating  rating_std  score_raw_avg  score_count_weighted  score_trust_weighted
B013SLWJJW            75    5.000000    0.000000       5.000000              2.548607              4.923727
B01B5BWTNS           494    4.846154    0.525820       4.846154              2.736944              4.857512
B014EB2ADA            65    4.923077    0.407266       4.923077              2.502931              4.857016
B0148B7EJ6           118    4.898305    0.330496       4.898305              2.524442              4.856693
B00RLSCLJM          2674    4.835079    0.567966       4.835079              4.115406              4.848120
B0006HB4XE           254    4.858268    0.544094       4.858268              2.590608              4.842190
B00JKV6EGE            88    4.863636    0.529190       4.863636              2.487406              4.827082
B00G8Q7JZ4           562    4.797153    0.722537       4

## 6. Summary

**Phase 8 Complete:**
- Aggregated review-level trust scores to product level
- Computed trust-weighted ratings
- Evaluated ranking quality with NDCG@K and Precision@K
- Compared against baselines
- Generated final product trust scores